<div style="text-align: center;">
    <h1><strong>Detección de <i>Clickbait</i> en noticias</strong></h1>
    <h2>Proyecto Procesado de Lenguaje Natural</h2>
    <br>
    <h3>Máster en Ciencia de Datos</h3>
    <h4>Universitat de València</h4>
    <br>
    <p>Juan Alcaráz Otón, Xueyao An, Fabián Calvo Castillo, Adrián Carrasco Alcalá, Javier Herrero Pérez, Mario Martínez Guillén y Clara Montalvá Barcenilla</p>
    <p><b>Curso 2025/2026</p>
</div>
<hr>

## Introducción

El periodismo digital ha transformado profundamente la forma en que los usuarios consumen información, generando un ecosistema altamente competitivo donde la atención del lector es el principal activo. 
En este contexto, ha proliferado el uso del clickbait, que esto se define como una estrategia utilizada en los medios digitales que busca llamar la atención a través de los titulares, apelando a las emociones y a la curiosidad de los lectores para forzar el click en la noticia, a menudo en perjuicio de la calidad informativa (Bravo Araujo, Serrano-Puche y Novoa Jaso, 2021).

De acuerdo con Bazaco et al. (2019), se trata de un fenómeno comunicativo dinámico que prioriza la interrogante sobre la información, omitiendo datos clave para generar un vacío de curiosidad o empleando un enfoque sensacionalista. Dentro del marco de este proyecto, se adoptará una aproximación dual para la identificación de clickbait:

* Por un lado, como un titular sensacionalista diseñado con estructuras lingüísticas concretas para captar la atención e incentivar el click, el cual puede ser detectado analizando únicamente el titular.

* Por otro lado, como un titular llamativo que presenta una diferencia considerable con el contenido de la noticia, ya sea porque plantea una pregunta que nunca se responde o porque presenta información completamnete opuesta a la que se desarrolla en el cuerpo del texto.

Para automatizar la detección de estos patrones, el proyecto emplea el modelo taniwasl/clickbait_es, alojado en Hugging Face. Este clasificador se fundamenta en BETO, la versión entrenada para el idioma español del modelo de lenguaje BERT (Bidirectional Encoder Representations from Transformers). Este modelo ha sido ajustado (fine-tuned) específicamente con un corpus aproximado de 30.000 noticias provenientes de múltiples medios españoles. 

Su funcionamiento radica en procesar la secuencia de palabras del titular analizando el contexto bidireccional de cada término mediante mecanismos de atención. Al haber aprendido de decenas de miles de ejemplos, el modelo es capaz de ponderar características semánticas y sintácticas intrínsecas del clickbait, permitiendo predecir y clasificar nuevos titulares con precisión.


## Objetivo

Objetivo General:

* Analizar y clasificar el uso de técnicas de clickbait en la prensa digital española mediante la extracción automatizada de noticias y la aplicación de técnicas de Procesamiento de Lenguaje Natural.

Objetivos Específicos:

1. Implementar técnicas de web scraping para adquirir de forma automatizada noticias de las categorías Internacional, Nacional y Cultura procedentes de una selección de los principales periódicos y medios de comunicación españoles.

2. Analizar los titulares y los contenidos textuales de las noticias desde la perspectiva del NLP para extraer características distintivas que diferencien las noticias con y sin clickbait, evaluando las tendencias en función del medio y la categoría.

3. Integrar un agente de Inteligencia Artificial que automatice la inferencia y el etiquetado masivo de los titulares adquiridos, clasificándolos en clickbait o no clickbait y con la capacidad añadida de generar un titular que adecuado a las noticias clasificadas como clickbait.


## Metodología

### 1. Adquisición de los Datos

Obtenemos noticias de los siguientes periódicos y medios españoles:

- ABC
- elDiario
- El Confidencial
- La Vanguardia
- 20minutos
- OkDiario
- RTVE
- Mediterráneo Digital
- El HuffPost

En particular, los periódicos digitales _Mediterráneo Digital_ y _El Huffpost_ destacan por su gran cantidad de titulares sensacionalistas y que no se corresponden con el contenido de los cuerpos de las noticias.

Extraemos los artículos de las siguientes categorías:

- Internacional: Noticias de ámbito mundial
- Nacional: Noticias de España y de sus regiones
- Cultura: Artes, cine, literatura, entretenimiento, etc.

Para ello, utilizamos tanto las páginas feed de aquellos periódicos que disponen de ellas como técnicas de scraping directo sobre los HTML de las páginas web.

Las noticias extraídas se almacenan en formato JSON con la siguiente estructura:

```json
{
  "Link": "string",
  "Periódico": "string",
  "Fecha": "string (YYYY-MM-DD)",
  "Título": "string",
  "Subtítulo": "string o null",
  "Categoría": "string",
  "Contenido": "string"
}
```

Los campos de esta estructura representan lo siguiente:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| Link | string | URL del artículo original |
| Periódico | string | Nombre del medio de comunicación |
| Fecha | string | Fecha de publicación (formato YYYY-MM-DD) |
| Título | string | Título principal del artículo |
| Subtítulo | string o null | Subtítulo o descripción breve |
| Categoría | string | Categoría o sección del artículo |
| Contenido | string | Texto completo del artículo |

Se crea un JSON para cada periódico, en el que se recogen todas sus noticias, y estos ficheros se guardan en la carpeta 'data/' con el formato de nombre 'nombredelmedio.json'.

### 2. Preprocesado del Texto

Una vez obtenidas suficientes noticias de los diferentes medios, las guardamos todas en el archivo JSON común 'conjunto_noticias.json'.

Tras una vista previa del contenido almacenado en dicho archivo, se determina que los elementos del texto que deberían eliminarse son los siguientes:

- **Comillas:** hay muchas formas diferentes que cada periódico utiliza para poner comillas, estas son: /""/, ««, '', "" y algunas más.  
- **Signos de interrogación y exclamación:** ¡! ¿?
- **Guiones:** --
- **URLs**
- **Direcciones de correo y cuentas de twitter:** empiezan por @
- **Saltos de línea:** \n \r
- **Signos de puntuación:** , ; : .
- **Corchetes y paréntesis:** [] ()
- **Emojis**

### 3. Clasificación de las Noticias en Clickbait/No Clickbait

Una vez finalizado el preprocesado del texto y consolidado el conjunto de datos limpio en conjunto_noticias.json, se procede a la etapa de clasificación automática de los titulares. Para esta tarea central, se implementa el modelo preentrenado taniwasl/clickbait_es, disponible en el repositorio de Hugging Face.

El procedimiento técnico de clasificación consta de los siguientes pasos:

1. Carga del entorno de inferencia: A través de la librería transformers, se instancia tanto el tokenizador correspondiente a la arquitectura BETO como el propio modelo de clasificación de secuencias (AutoModelForSequenceClassification).

2. Tokenización de titulares: Cada valor del campo "Título" de nuestro JSON se pasa por el tokenizador. Este paso convierte el texto crudo en tensores y añade los tokens especiales de inicio y fin de oración que requiere la red neuronal para acotar el contexto.

3. Inferencia del modelo: Los tensores se introducen en el modelo preentrenado. Gracias a su fine-tuning previo sobre noticias de la prensa española, el modelo hace pasar las representaciones vectoriales por sus distintas capas ocultas. El sistema detecta patrones típicos del sensacionalismo en español como la hiperbolización, deícticos temporales o preguntas abiertas.

4. Etiquerado: La red neuronal devuelve unos valores que determinan la probabilidad de que el texto pertenezca a la clase clickbait o a la clase no clickbait. A cada noticia en el conjunto de datos se le adjunta esta nueva etiqueta predictiva.

La automatización de este proceso nos permite generar la variable objetivo para todo el conjunto de noticias. Con los datos ya etiquetados, es posible realizar un análisis cruzado para validar estadísticamente qué medios y categorías temáticas incurren con mayor frecuencia en la desinformación o en tácticas de clickbait.


### 4. Extracción de características

### 5. Implementación Agéntica

## Resultados

## Conclusiones

## Bibliografía

Bazaco, A., Redondo, M., y Sánchez-García, P. (2019). El clickbait como estrategia del periodismo viral: concepto y metodología. Revista Latina de Comunicación Social, (74), 94-115.

Bravo Araujo, A., Serrano-Puche, J., y Novoa Jaso, M. (2021). Uso del clickbait en los medios nativos digitales españoles. Un análisis de El Confidencial, El Español, Eldiario.es y Ok Diario. Doxa Comunicación. Revista Interdisciplinar de Estudios de Comunicación y Ciencias Sociales, (32), 185-210.

Taniwa (2023). taniwasl/clickbait_es. Hugging Face. Recuperado de https://huggingface.co/taniwasl/clickbait_es


- *Información sobre los periódicos analizados*

